# Single-Engine OpenVINO Grounder - Cold/Warm Latency Remeasure

**Author:** kj
**Approach:** time the live single-engine OpenVINO int8 grounder per claim, under the `LATENCY` compile hint, in two regimes - cold and warm.

The cold/warm latency table in `docs/experiments/semantic-grounding-sota.md` predates the `compile_ir` default flip from `THROUGHPUT` to `LATENCY` (the latter is ~2x faster for inline single-claim serving). This notebook remeasures both regimes directly so the documented numbers reflect the shipped default. The two cross-encoders (reranker + NLI) on the top-k survivors dominate once embedding is cached, so the gap between regimes isolates the chunk-embedding cost.

**Phases:**
1. **Load** the three published HF int8 IRs (embedder, reranker, NLI) - `compile_ir` now defaults to `LATENCY`
2. **Cache** every unique chunk's embedding once (the warm path reuses these; the cold path re-embeds per claim)
3. **Time** cold and warm per claim across `top-k` in [5, 8, 12, 50], then the k=8 per-claim distribution (median / p90)

**Output:** a cold/warm ms-per-claim table by top-k and the k=8 distribution, ready to refresh the SOTA latency section. Read-only - no artifacts written.

## Environment

CPU-only - OpenVINO serves the int8 IRs on CPU; no CUDA. Set before any model import so the runtime never probes a GPU.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""          # CPU-only (OpenVINO int8)
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # silence HF fork warning
os.environ["HF_HUB_DISABLE_XET"] = "1"           # plain HTTP downloads from the Hub

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import time
from pathlib import Path

# Data
import numpy as np

# Rich console output + progress
from rich import print as rprint
from tqdm.auto import tqdm

# Project - single-engine OpenVINO grounder helpers
from grounding_models import load_gold
from grounding_openvino import (
    load_ov_hf, embed_vectors, topk_chunks, rerank_max, nli_max, HF_REPOS,
)

## Reproducibility

Fixed seed so the timed record sample is identical across runs.

In [ ]:
SEED = 42
np.random.seed(SEED)

## Configuration

Timing knobs only - no model hyperparameters. `N` records are drawn from the gold and timed in both regimes for every `top-k`; the deployed width is `TOP_K`. The pool is CLS (the bge / mDeBERTa serving convention).

In [ ]:
# Pipeline
TOP_K = 8                                 # deployed pre-filter width
K_SWEEP = [5, 8, 12, 50]                  # 50 == all chunks (no pre-filter)
POOL = "cls"                              # bi-encoder pooling

# Timing
N = 30                                    # records timed per regime per k
WARMUP = 3                                # untimed warm-up iterations

# Runtime
import openvino as ov
HINT = "LATENCY"                          # compile_ir default; THROUGHPUT only helps batch

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"-" * 44}[/dim]
[bold]Pipeline[/bold]
  Deployed top-k: [yellow]{TOP_K}[/yellow] [dim](sweep {K_SWEEP})[/dim]
  Pool: [cyan]{POOL}[/cyan]

[bold]Timing[/bold]
  Records per regime/k: [yellow]{N}[/yellow]
  Warm-up iters: [yellow]{WARMUP}[/yellow]

[bold]Runtime[/bold]
  Device: [green]CPU[/green] [dim](OpenVINO {ov.__version__})[/dim]
  Compile hint: [green]{HINT}[/green]
""")

## Data Loading

Loads the verified gold and draws a fixed `N`-record sample to time. The typical claim carries the full evidence set (chunks/claim median 50), so the sample is representative of typical-case, not worst-case, latency.

In [ ]:
recs = load_gold()
idx = np.random.choice(len(recs), N, replace=False)
sample = [recs[i] for i in idx]
median_chunks = int(np.median([len(r["chunks"]) for r in sample]))

rprint(f"""[white]Gold loaded[/white]
  Records: [yellow]{len(recs)}[/yellow]  timed sample: [yellow]{N}[/yellow]
  Chunks/claim (sample median): [yellow]{median_chunks}[/yellow]
""")

## Load int8 models and build the warm cache

Pulls the three published OpenVINO int8 IRs from the Hub (`compile_ir` compiles each with the `LATENCY` hint). The warm cache then embeds every unique chunk in the sample exactly once, keyed by content - this is what the warm path reuses instead of re-embedding per claim.

In [ ]:
ecm, etok, _ = load_ov_hf("bge-m3")                 # bi-encoder pre-filter
rcm, rtok, _ = load_ov_hf("bge-reranker-v2-m3")     # cross-encoder relevance
ncm, ntok, _ = load_ov_hf("mDeBERTa-v3-nli")        # cross-encoder NLI

# Warm cache: embed each unique chunk once, key by content
uniq = list({c for r in sample for c in r["chunks"]})
cvecs = embed_vectors(ecm, etok, uniq, pool=POOL)
cache = {c: v for c, v in zip(uniq, cvecs)}

rprint(f"""[white]Models compiled[/white] [dim](LATENCY hint)[/dim]
  Embedder: [cyan]{HF_REPOS['bge-m3']}[/cyan]
  Reranker: [cyan]{HF_REPOS['bge-reranker-v2-m3']}[/cyan]
  NLI: [cyan]{HF_REPOS['mDeBERTa-v3-nli']}[/cyan]

[white]Warm cache[/white]
  Unique chunks embedded once: [yellow]{len(uniq)}[/yellow]
""")

## Latency regimes

Two per-claim functions sharing the same cross-encoder scoring; they differ only in how the top-k chunks are obtained:

- **Cold** - `topk_chunks` re-embeds every chunk for this claim, then ranks. First time a chunk is seen
- **Warm** - only the claim is embedded; the cosine rank reuses the cached chunk vectors. Every subsequent claim reusing seen chunks

Both then score the top-k survivors with the reranker and the NLI. The max-over-chunks gate means only the survivors matter.

In [ ]:
def cold_one(r, k):
    """Cold: re-embed all chunks (topk_chunks) then score top-k with both cross-encoders."""
    ch = topk_chunks(ecm, etok, r["claim"], r["chunks"], k, pool=POOL)
    rerank_max(rcm, rtok, r["claim"], ch)
    nli_max(ncm, ntok, r["claim"], ch)


def warm_one(r, k):
    """Warm: embed only the claim; rank via cached chunk vectors; score top-k."""
    chunks = r["chunks"]
    if k is None or k >= len(chunks):
        ch = chunks
    else:
        cv = embed_vectors(ecm, etok, [r["claim"]], pool=POOL)[0]
        dv = np.stack([cache[c] for c in chunks])
        order = np.argsort(-(dv @ cv))[:k]
        ch = [chunks[i] for i in sorted(order)]
    rerank_max(rcm, rtok, r["claim"], ch)
    nli_max(ncm, ntok, r["claim"], ch)


def time_regime(fn, records, k):
    """Per-claim wall time in ms for `fn` over `records` at top-k `k`."""
    per = []
    for r in records:
        t = time.time(); fn(r, k); per.append((time.time() - t) * 1000)
    return np.array(per)


# Warm up the kernels (untimed) so the first timed batch is not cold-start biased
for _ in range(WARMUP):
    cold_one(sample[0], TOP_K)
    warm_one(sample[0], TOP_K)

## Run the sweep

Times both regimes for every `top-k`. The cold all-chunks point (k=50) is the slowest (~10 s/claim), so the bar advances slowly there; `mininterval` rate-limits refreshes to avoid flooding the kernel.

In [ ]:
dist = {}
for k in tqdm(K_SWEEP, desc="top-k sweep", mininterval=2.0):
    cold = time_regime(cold_one, sample, k)
    warm = time_regime(warm_one, sample, k)
    dist[k] = (cold, warm)

## Summary

The cold/warm ms-per-claim table by top-k, and the k=8 per-claim distribution (median / p90). Warm is tight and roughly chunk-count-independent because only the top-k pairs are scored; cold scales with chunk count because the pre-filter re-embeds everything.

In [ ]:
rows = "\n".join(
    f"  top-k [yellow]{k:>2}[/yellow]: cold [cyan]{dist[k][0].mean():6.0f}[/cyan] ms   "
    f"warm [green]{dist[k][1].mean():6.0f}[/green] ms"
    + ("  [dim]<- deployed[/dim]" if k == TOP_K else "")
    for k in K_SWEEP
)
c8, w8 = dist[TOP_K]
speedup = np.median(c8) / np.median(w8)

rprint(f"""[bold cyan]Cold/warm latency vs top-k ({N} records, CPU, LATENCY hint)[/bold cyan]
[dim]{"-" * 56}[/dim]
{rows}

[bold]k={TOP_K} per-claim distribution[/bold]
  cold  median [cyan]{np.median(c8):6.0f}[/cyan] ms   p90 [cyan]{np.percentile(c8, 90):6.0f}[/cyan] ms
  warm  median [green]{np.median(w8):6.0f}[/green] ms   p90 [green]{np.percentile(w8, 90):6.0f}[/green] ms
  warm speedup (median): [green]{speedup:.1f}x[/green]
""")